# Scraping WDumper data

<a href="https://colab.research.google.com/github/itamargiv/wdumper-scraper/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In order to better understand what kinds of entity data dump subsets our users are interested in, this repository scrapes all dump subsets listed under "recent dumps". The scrape includes a JSON representation of the filters that were used to generate the dump.

The notebook generates a csv file that includes filter data in a human-readable form. Each row of the csv includes the following columns:
- dump name
- URL
- filter (in human-readable form including labels for any items and properties used)
- statements included in the dump (in human-readable form)
- labels (yes/no)
- descriptions (yes/no)
- aliases (yes/no)
- sitelinks (yes/no)
- languages

## Setup Environment

This notebook can be run on Google  (Colab). To set up the environment in Colab, we need to clone the repository and install the required dependencies.

In [ ]:
import os
import sys

REPO_NAME = 'wdumper-scraper'
REPO_URL = f'https://github.com/itamargiv/{REPO_NAME}.git'
REPO_PATH = f'/content/{REPO_NAME}'

if 'google.colab' in sys.modules:
    print('Running on Google Colab...')
    if os.path.exists(REPO_PATH):
        !rm -rf {REPO_PATH}

    !git clone --depth 1 {REPO_URL}
    !pip install /content/{REPO_NAME}
else:
    print('Running on local machine...')

print('Setup complete.')

## Scrape Dump Data

The code snippet below scrapes all dumps listed under "recent dumps". It uses a thread pool to speed up the scraping process, and it handles any exceptions that may occur during scraping. The scraped data is stored in a list of dictionaries, which can then be converted to a csv file for further analysis.

In [ ]:
from pprint import pprint
from requests_cache import CachedSession
from wdumper_scraper import Scraper, DumpsInfoLoader

session = CachedSession(
    'scraper_cache',
    backend='sqlite'
)

scraper = Scraper(session, log = True)
dumps_loader = DumpsInfoLoader(scraper)
results = dumps_loader.scrape()
error_msgs = [data["error"] for data in results.skipped]
error_counts = {msg: error_msgs.count(msg) for msg in set(error_msgs)}

print(f'Scraped {len(results.dumps)} dumps, skipped {len(results.skipped)} dumps.')
print(f'Error counts for skipped dumps:')
pprint(error_counts)